In [6]:
##### importing custom modules from the projects folder
import sys
from pathlib import Path
# Start at current working directory
current = Path.cwd()
# Walk up the tree until config.py is found or root is reached
for parent in [current] + list(current.parents):
    config_path = parent / "config.py"
    if config_path.exists():
        sys.path.append(str(parent))
        import config # <<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<< 
        break
else:
    raise FileNotFoundError("config.py not found in any parent directories")

import scripts.scrapers.nbaScraper as ns

#import actNetScraper as ans
import scripts.scrapers.actNetApi as ans
import scripts.scrapers.nbApi as nbapi
from datetime import datetime, timedelta
# -----------------------------------------------
# -------------------------------- PARAMS
leagues =  ['nhl', 'nba']  # None or list ['nba', 'nhl', 'nfl', 'mlb', 'wnba'] # NONE looks for all sports
specified =  []  ####  [specific prop] or [] for all props, *****only works with a single league in leagues

# day adjustment from today (date of running script), negative = dates into the past
dayJump = 0
# date can be a list of dates if multiple need scraping 'YYYY-MM-DD'
# default is to only pull today or today + dayJump
dates = [(datetime.today() + timedelta(days=dayJump)).strftime('%Y-%m-%d')]
#dates = ['2025-10-13']#,  '2025-10-02']#, '2025-09-13']

database_export = True  # add all scrapes to database
store_locally = True    # add all scrapes to class variables
season_int = 2026 # int will be the final year of the schedule season
season_str = '2025-26'
season_start_date = '2025-10-21' #'MM/DD/YYY'
season_type = 'Regular+Season' # ['Regular+Season', 'PlayIn', 'Playoffs']
per_mode = 'Totals' #['Totals', 'PerGame']
# ------------------------------------------------


In [4]:
# nba website and basketball referenece scrapers
scraper = ns.scraper(
    browser_path = str(config.BROWSER_DIR) + '\\geckodriver.exe',
    database_export = database_export, 
    store_locally = store_locally,
    pymysql_conn_str =  None
)
# assigns the date of the last time code executed as today
today = scraper.meta_data['today_dt']

# looks up the actual date for the last regular season game date. this will be used to grab the data for the completed games on the date
nbaApi = nbapi.nbaApi(
    browser_path = str(config.BROWSER_DIR) + '\\geckodriver.exe',
    database_export = database_export, 
    store_locally = store_locally,
    pymysql_conn_str =  None
)
a = nbaApi.get_last_game_date(season = season_str)
run_date = nbaApi.last_game_date
dateRange = [
    run_date, run_date
]

#prop_scraper = ans.actNetScraper(
prop_scraper = ans.actNetApi(    
    browser_path = str(config.BROWSER_DIR) + '\\geckodriver.exe',
    dates = dates,
    leagues = leagues,
    database_export = database_export, 
    store_locally = store_locally,
    second_run = False
)

# turn to False if issues loading new players
#prop_scraper.update_players = False

# update the league list to only ones with games today
#leagues = prop_scraper.check_for_league_games(date_check = None, update_class_leagues_var = True)
print('scraping for', prop_scraper.leagues, 'on', prop_scraper.dates)

Last day with games played: 2025-10-21
scraping for ['nhl', 'nba'] on ['2025-10-22']


In [5]:
# SCRAPE PROPS
print(today, 'run date...\n for', dates, 'and', prop_scraper.leagues)
############## these 2 lines were used w. old selenium setup in actNetScraper.py
#prop_scraper.scrape(sleep_secs = 3, specific_props = specified,leagues_override = prop_scraper.leagues,an_state_code = 'BC')
#prop_scraper.processScrapes(remove_dups = True,specific_props = specified)
###############
prop_scraper.scrape(
    sleep_secs = 3, 
    specific_props = specified,
    leagues_override = prop_scraper.leagues,
    an_state_code = 'BC'
)
print('html saved...\n')

prop_scraper.processScrapes(
    remove_dups = True,
    specific_props = specified
) 

if prop_scraper.scrape_error_flag:
    print(prop_scraper.scrape_errors)
    #prop_scraper.tryMissingProps()


2025-10-22 run date...
 for ['2025-10-22'] and ['nhl', 'nba']
html saved...

processing nhl ...
original rows:  (553, 20)
after dups removed:  (553, 20)
['Colten Ellis']
nhl odds data loaded...
prop          ast  ats  gs  gs1st  gs2plus  gs3plus  gsLast  pts  sog
propId count   50  110   6    111       48       23     110   50   45
processing nba ...
original rows:  (1942, 20)
after dups removed:  (1942, 20)
['Ryan Kalkbrenner' 'Walter Clayton' 'Yang Hansen' 'Tre Johnson'
 'Ace Bailey' 'Kon Knueppel' 'VJ Edgecombe' 'Dylan Harper' 'Cooper Flagg'
 'Ariel Hukporti' 'Collin Gillespie' 'Derrick Jones Jr.' 'Craig Porter']
nba odds data loaded...
prop          ast  blk   pa   pr  pra  pts   ra  reb   sb  stl  threes
propId count  187  174  173  180  184  187  176  187  154  170     170


In [19]:
# SCRAPE BREF TEAM MISC
print(today, 'run date...\n')
#teams = ['GSW','DEN','POR','SAC','TOR','DAL','PHO','CHI','LAL','HOU','MIA','MEM','DET','MIL','NOP','MIN','CLE','OKC','LAC','BRK','SAS','NYK','WAS','CHO','UTA','IND','BOS','PHI','ATL','ORL']
scraper.get_bref_pos_estimates(
        base_url = 'https://www.basketball-reference.com/teams/{team}/{season}.html#pbp', 
        today_date = today,
        season = season_int,
        database_table = 'brefmisc',
        team_overrides = None
)
### CAN DELETE AFTER CONFIRMING ERROR RETRIES IN FUNCTION WORK AS EXPECTED
missing_teams = scraper.scrape_errors['brefmisc']['url']
if len(missing_teams):
        print('missing:', missing_teams)
        for i in missing_teams:
                scraper.get_bref_pos_estimates(
                        base_url = 'https://www.basketball-reference.com/teams/{team}/{season}.html#pbp', 
                        today_date = today,
                        season = season_int,
                       database_table = 'brefmisc',
                        team_overrides = [i[0]]
               )


2025-06-22 run date...

bref player pos estimates scraped, 30 teams...


In [7]:
# NBA API hits
# ---------------------------------------------------------------- #
# --- TEAM --- #
print(today, 'run date...\n')
nbaApi.request_nba_team_shotzone_data(
    season = season_str, #'YYYY-YY'
    start_date = run_date, # will force to pd.datetime.date()
    end_date = run_date, 
    per_mode = per_mode, #['Totals', 'PerGame'] 
    season_type = season_type, #['Regular+Season', 'PlayIn', 'Playoffs']
    distance_type = 'By+Zone', # ['8ft+Range', '5ft+Range','By+Zone']
    sob = ['Base', 'Opponent'], #['Base', 'Opponent']  base= teams offense, opponent = teams defense
    database_table = 'statsteamshotzones'     
)

# PER GAME
nbaApi.request_nba_team_stats(
    season = season_str, #'YYYY-YY'
    start_date = season_start_date, # will force to pd.datetime.date()
    end_date = run_date, 
    per_mode = 'PerGame', #['Totals', 'PerGame'] 
    season_type = season_type, #['Regular+Season', 'PlayIn', 'Playoffs']
    measure_type = ['Base', 'Advanced', 'Opponent'], #['Base', 'Advanced', 'Opponent']  
    database_table = 'statsteam'
)
# TOTALS
nbaApi.request_nba_team_stats(
    season = season_str, #'YYYY-YY'
    start_date = run_date, # will force to pd.datetime.date()
    end_date = run_date, 
    per_mode = 'Totals', #['Totals', 'PerGame'] 
    season_type = season_type, #['Regular+Season', 'PlayIn', 'Playoffs']
    measure_type = ['Base', 'Advanced', 'Opponent'], #['Base', 'Advanced', 'Opponent']  
    database_table = 'statsteamtotals'
)

# ---------------------------------------------------------------- #
# --- PLAYER --- #
print(today, 'run date...\n')
nbaApi.request_nba_player_shotzone_data(
            season = season_str, #'YYYY-YY'
            start_date = run_date, # will force to pd.datetime.date()
            end_date = run_date, 
            per_mode = per_mode, #['Totals', 'PerGame'] 
            season_type = season_type, #['Regular+Season', 'PlayIn', 'Playoffs']
            distance_type = 'By+Zone', # ['8ft+Range', '5ft+Range','By+Zone'],
            sob = ['Base'], #['Base', 'Opponent']  base= teams offense, opponent = teams defense
            database_table = 'statsplayershotzones'        
)
# Passing
nbaApi.request_nba_player_tracking(
        season = season_str, #'YYYY-YY'
        start_date =run_date, # will force to pd.datetime.date()
        end_date = run_date, 
        per_mode = per_mode, #['Totals', 'PerGame'] 
        season_type = season_type, #['Regular+Season', 'PlayIn', 'Playoffs']
        measure_type = 'Passing', #['Passing','Rebounding','Drives','Possessions','Efficiency','PostTouch', 'ElbowTouch', 'PaintTouch']  
        database_table = 'statsplayerpassing'      
)
# Rebounding
nbaApi.request_nba_player_tracking(
        season = season_str, #'YYYY-YY'
        start_date =run_date, # will force to pd.datetime.date()
        end_date = run_date, 
        per_mode = per_mode, #['Totals', 'PerGame'] 
        season_type = season_type, #['Regular+Season', 'PlayIn', 'Playoffs']
        measure_type = 'Rebounding', #['Passing','Rebounding','Drives','Possessions','Efficiency']  
        database_table = 'statsplayerrebounding'      
)

# add these measure types - 
#### 'Drives', 'Possessions', 'Efficiency' 
# maybe add these - 'PostTouch', 'ElbowTouch', 'PaintTouch'
#nbaApi.request_nba_player_tracking()

2025-10-22 run date...

nba team shot zone retrieved 4 offensive,  4 defensive loaded...
nba team stats PerGame retrieved (4, 65) loaded...
nba team stats Totals retrieved (4, 65) loaded...
2025-10-22 run date...

nba player shot zone retrieved 40 offensive loaded...
nba player Passing retrieved 40 loaded...
nba player Rebounding retrieved 33 loaded...


In [ ]:
'''
These 2 playtype hits require some games before they populate on the site 
https://www.nba.com/stats/teams/playtype-post-up  check this for stats

'''

print(today, 'run date...\n')
# --- TEAM --- #
nbaApi.request_nba_team_playtype_data(
            season = season_str, #'YYYY-YY' 
            play_type = [
                'Isolation', 'Transition','PRBallHandler','PRRollman', 'Postup', 'Spotup', 
                'Handoff', 'Cut', 'OffScreen', 'OffRebound', 'Misc'
            ], 
            lid='00',
            per_mode = 'PerGame', #['Totals', 'PerGame'] 
            season_type = season_type, #['Regular+Season', 'PlayIn', 'Playoffs']
            sob = ['offensive', 'defensive'],
            sleep_time = 2,
            database_table = 'statsteamplaytypes'
        )


print(today, 'run date...\n')
nbaApi.request_nba_player_playtype_data(
            season = season_str, #'YYYY-YY' 
            play_type = [
                'Isolation', 'Transition','PRBallHandler','PRRollman', 'Postup', 'Spotup', 
                'Handoff', 'Cut', 'OffScreen', 'OffRebound', 'Misc'
            ], 
            lid='00',
            per_mode = 'PerGame', #['Totals', 'PerGame']
            season_type = season_type, #['Regular+Season', 'PlayIn', 'Playoffs']
            sob = ['offensive'], #'offensive','defensive'],
            sleep_time = 2,
            database_table = 'statsplayerplaytypes'
)


# scratch

In [8]:
from nba_api.stats.endpoints import leaguegamefinder 
# Fetch all games for the current season
seasons = '2025-26'
gamefinder = leaguegamefinder.LeagueGameFinder(season_nullable=seasons, league_id_nullable='00')
g = gamefinder.get_data_frames()[0]
g

,SEASON_ID,TEAM_ID,TEAM_ABBREVIATION,TEAM_NAME,GAME_ID,GAME_DATE,MATCHUP,WL,MIN,PTS,...,FT_PCT,OREB,DREB,REB,AST,STL,BLK,TOV,PF,PLUS_MINUS
0,22025,1610612745,HOU,Houston Rockets,0022500001,2025-10-21,HOU @ OKC,L,292,124,...,0.871,16,36,52,23,6,5,22,26,-1.0
1,22025,1610612744,GSW,Golden State Warriors,0022500002,2025-10-21,GSW @ LAL,W,241,119,...,0.897,9,31,40,29,10,4,18,27,10.0
2,22025,1610612760,OKC,Oklahoma City Thunder,0022500001,2025-10-21,OKC vs. HOU,W,290,125,...,0.800,11,27,38,29,12,4,11,27,1.0
3,22025,1610612747,LAL,Los Angeles Lakers,0022500002,2025-10-21,LAL vs. GSW,L,240,109,...,0.607,7,32,39,23,7,2,19,21,-10.0
4,12025,1610612766,CHA,Charlotte Hornets,0012500066,2025-10-17,CHA @ NYK,L,240,108,...,0.727,10,31,41,30,9,3,21,20,-5.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
144,12025,1610612756,PHX,Phoenix Suns,0012500001,2025-10-03,PHX @ LAL,W,241,103,...,0.538,10,40,50,29,9,5,16,31,22.0
145,12025,1610612747,LAL,Los Angeles Lakers,0012500001,2025-10-03,LAL vs. PHX,L,240,81,...,0.784,11,35,46,10,9,7,22,25,-22.0
146,12025,15016,MEL,Melbourne United,0012500009,2025-10-03,MEL @ NOP,L,240,97,...,0.650,13,35,48,23,7,1,17,18,-10.2
147,12025,1610612752,NYK,New York Knicks,0012500008,2025-10-02,PHI @ NYK,W,238,99,...,0.656,20,38,58,19,13,2,17,26,15.0
